# 1b · Augmentation de Texto con LLM (Claude API)

> **Pipeline:** `1_Sobremuestreo.ipynb` → **`1b_Augmentation_LLM.ipynb`** → `2_Modelos_Tradicionales_editado.ipynb`

| Patología | Reales | Ratio | Estrategia |
|-----------|--------|-------|------------|
| `hemorragia` | 3319 | 1:0.2 | Sin modificar — clase dominante |
| `acv` | 1751 | 1:1.3 | ROS (notebook 1) |
| `desviacion_linea_media` | 52 | 1:75 | **LLM augmentation — este notebook** |
| `fractura_compleja_craneo` | 60 | 1:65 | **LLM augmentation — este notebook** |

**Referencia:** Veselovsky et al. (2023) — *Artificial Intelligence with Purpose.* Stanford HAI.

## 0 · Instalación

In [1]:
# Ejecuta esta celda solo la primera vez
#%pip install anthropic python-dotenv openpyxl scikit-learn --quiet

## 1 · Configuración de la API

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
RANDOM_SEED       = int(os.getenv("RANDOM_SEED", 42))

if not ANTHROPIC_API_KEY:
    raise EnvironmentError(
        "No se encontró ANTHROPIC_API_KEY en el .env\n"
        "Agrégala como: ANTHROPIC_API_KEY=sk-ant-..."
    )

print(f"API key cargada: {ANTHROPIC_API_KEY[:18]}...")

API key cargada: sk-ant-api03-Pn8SX...


In [3]:
import anthropic
import pandas as pd
import numpy as np
import random
import time
import json
import re
import threading
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print(f"anthropic SDK: {anthropic.__version__}")
print("Cliente Claude inicializado.")

anthropic SDK: 0.97.0
Cliente Claude inicializado.


## 2 · Carga y etiquetado de datos

⚠️ **Siempre desde `rad_criticos.xlsx` original** — nunca desde el output del notebook 1 para evitar que duplicados ROS contaminen los ejemplos de referencia.

In [4]:
RAD_PATH = Path('../data/rad_criticos.xlsx').resolve()
if not RAD_PATH.exists():
    raise FileNotFoundError(f"No encontrado: {RAD_PATH}")

df = pd.read_excel(RAD_PATH)
df['Hallazgos'] = df['Hallazgos'].fillna('').astype(str)
df['Opinión']   = df['Opinión'].fillna('').astype(str)

print(f"Dataset: {RAD_PATH.name}  ({len(df)} registros)")

Dataset: rad_criticos.xlsx  (952 registros)


In [5]:
def detectar_patologia_robusta(texto, keywords_positivos, keywords_negativos=None):
    if not isinstance(texto, str):
        return 0
    texto = texto.lower()
    keywords_negativos = keywords_negativos or [
        'antiguo','antigua','cronico','cronica','crónico','crónica',
        'viejo','vieja','residual','residuales','secuela','secuelar','secuelas',
        'previo','previa','previos','previas','conocido','conocida',
        'sin cambios','estable','estables','resolucion','resolución'
    ]
    negaciones_contextuales = [
        'sin','descarta','descartado','descartada','negativo','negativa',
        'ausencia','ausente','ausentes','sin evidencia','sin signos',
        'no se observa','no se observan','no se evidencia','no se evidencian',
        'no se identifica','no se identifican','libre de',
        'negativo para','negativa para'
    ]
    keywords_ordenadas = sorted(keywords_positivos, key=len, reverse=True)
    oraciones = re.split(r'[.;]', texto)
    for oracion in oraciones:
        oracion = oracion.strip()
        kw = next((k for k in keywords_ordenadas if k in oracion), None)
        if not kw: continue
        if re.search(re.escape(kw) + r'\s*:\s*no\b', oracion): continue
        if re.search(re.escape(kw) + r'\s*:[^.;]*?\b(no|negativo|negativa|ausente)\b', oracion): continue
        idx = oracion.find(kw)
        if any(n in oracion[max(0,idx-50):idx] for n in negaciones_contextuales): continue
        if re.search(r'sin\s+(hallazgos?|evidencia|signos?)\s+(indirectos?\s+)?(de\s+)?' + re.escape(kw), oracion): continue
        if any(a in oracion for a in keywords_negativos): continue
        if 'ya conocido' in oracion or 'ya conocida' in oracion: continue
        if 'sin cambios' in oracion: continue
        if 'resolucion' in oracion or 'resolución' in oracion: continue
        return 1
    return 0

KEYWORDS = {
    'acv': ['infarto cerebral','infarto isquemico','accidente cerebrovascular',
            'hipodensidad sugestiva de isquemia','zona de isquemia',
            'area de isquemia','evento isquemico','ictus','acv','isquemia'],
    'hemorragia': ['hemorragia subaracnoidea','hemorragia intraparenquimatosa',
                   'hemorragia intraventricular','hematoma subdural','hematoma epidural',
                   'hematoma intraparenquimatoso','sangrado intraparenquimatoso',
                   'coleccion hematica','hemorragia','hematoma','sangrado'],
    'desviacion_linea_media': ['desviacion de la linea media','desviacion linea media',
                               'linea media desplazada','desplazamiento de la linea media',
                               'efecto de masa con desviacion','shift de linea media',
                               'herniacion subfalcina','hernia subfalcina'],
    'fractura_compleja_craneo': ['fractura con hundimiento','fractura hundimiento',
                                 'fractura deprimida','fractura conminuta','fractura compleja',
                                 'fractura de base de craneo','fractura craneofacial',
                                 'fractura multiple de craneo']
}
PATOLOGIAS = ['acv', 'hemorragia', 'desviacion_linea_media', 'fractura_compleja_craneo']

texto_etiquetado = df['Hallazgos'] + ' ' + df['Opinión']
for pat, kws in KEYWORDS.items():
    df[pat] = texto_etiquetado.apply(lambda t: detectar_patologia_robusta(t, kws))

print("Distribución de etiquetas (dataset original):")
print(f"  {'Patología':<30} {'Positivos':>10} {'%':>7}")
print("  " + "-"*52)
for p in PATOLOGIAS:
    n = int(df[p].sum())
    print(f"  {p:<30} {n:>10} {100*n/len(df):>6.1f}%")

Distribución de etiquetas (dataset original):
  Patología                       Positivos       %
  ----------------------------------------------------
  acv                                   323   33.9%
  hemorragia                            813   85.4%
  desviacion_linea_media                129   13.6%
  fractura_compleja_craneo               30    3.2%


## 3 · Parámetros de augmentation

**Solo cambia esta celda** para controlar el comportamiento completo.

In [6]:
# ══════════════════════════════════════════════════════════════════
#  PARÁMETROS — ajusta aquí, no en otras celdas
# ══════════════════════════════════════════════════════════════════

# Cuántos sintéticos ADICIONALES generar por clase
# (encima de los casos reales existentes)
N_SINTETICOS_POR_CLASE = 300

# Modelo Claude
# claude-haiku-4-5-20251001   → más barato (~$0.40 para 600 reportes)
# claude-sonnet-4-20250514    → mejor calidad (~$2.70 para 600 reportes)
MODELO = 'claude-haiku-4-5-20251001'

# Temperatura alta = mayor diversidad textual (recomendado por el paper)
TEMPERATURA = 0.9

# Máximo de llamadas por minuto (Claude Haiku: ~50 RPM en tier 1)
# Pon 25 para ser conservador y no recibir 429s
RPM_LIMIT = 25

# Umbral TF-IDF para detectar copias de reales
# 0.90 = equilibrio; sube a 0.95 si ves muchos rechazos 'similar_a_real'
UMBRAL_SIMILITUD = 0.90

# Clases a augmentar (no tocar hemorragia ni acv)
CLASES_A_AUGMENTAR = ['desviacion_linea_media', 'fractura_compleja_craneo']

# ── Resumen del plan ──────────────────────────────────────────────
total_llamadas = N_SINTETICOS_POR_CLASE * len(CLASES_A_AUGMENTAR)
t_min = total_llamadas / RPM_LIMIT

PRECIOS = {
    'claude-haiku-4-5-20251001':  (1.0,  5.0),
    'claude-sonnet-4-20250514': (3.0, 15.0),
}
p_in, p_out = PRECIOS.get(MODELO, (3.0, 15.0))
tokens_in  = total_llamadas * 420
tokens_out = total_llamadas * 200
costo_est  = (tokens_in / 1e6) * p_in + (tokens_out / 1e6) * p_out

print("PLAN DE AUGMENTATION")
print("=" * 60)
for clase in CLASES_A_AUGMENTAR:
    n_real = int(df[clase].sum())
    print(f"  {clase}")
    print(f"    Reales actuales : {n_real}")
    print(f"    Sintéticos meta : {N_SINTETICOS_POR_CLASE}")
    print(f"    Total esperado  : {n_real + N_SINTETICOS_POR_CLASE}")
print()
print(f"  Modelo      : {MODELO}")
print(f"  Temperatura : {TEMPERATURA}")
print(f"  RPM_LIMIT   : {RPM_LIMIT}")
print(f"  Llamadas    : {total_llamadas}")
print(f"  Tiempo est. : ~{t_min:.0f} min  (ambas clases en paralelo: ~{t_min/2:.0f} min)")
print(f"  Costo est.  : ~${costo_est:.2f} USD")

PLAN DE AUGMENTATION
  desviacion_linea_media
    Reales actuales : 129
    Sintéticos meta : 300
    Total esperado  : 429
  fractura_compleja_craneo
    Reales actuales : 30
    Sintéticos meta : 300
    Total esperado  : 330

  Modelo      : claude-haiku-4-5-20251001
  Temperatura : 0.9
  RPM_LIMIT   : 25
  Llamadas    : 600
  Tiempo est. : ~24 min  (ambas clases en paralelo: ~12 min)
  Costo est.  : ~$0.85 USD


## 4 · Prompt, rate limiter y función de generación

In [7]:
DESCRIPCION_CLINICA = {
    'desviacion_linea_media': (
        "desplazamiento de estructuras de la línea media cerebral (septum pellucidum, "
        "tercer ventrículo, glándula pineal) hacia un lado, causado por efecto de masa "
        "unilateral (hematoma, tumor, edema). Se mide en mm desde la línea media anatómica. "
        "Puede asociarse a herniación subfalcina."
    ),
    'fractura_compleja_craneo': (
        "fractura craneal compleja: hundimiento (fragmentos por debajo de tabla interna), "
        "conminución (múltiples fragmentos), fractura de base de cráneo (fosa anterior, media "
        "o posterior), o fractura craneofacial. Frecuente en trauma de alta energía."
    )
}

META_FRASES = [
    'aquí tienes','aqui tienes','claro,','claro!','como solicitaste',
    'informe radiológico sintético','informe radiologico sintetico',
    'aquí está','aqui esta','con gusto','a continuación te presento',
    'por supuesto','entendido,','como pediste','por supuesto,'
]


def construir_prompt(hallazgos_ejemplo: str, opinion_ejemplo: str, clase: str) -> str:
    desc   = DESCRIPCION_CLINICA[clase]
    nombre = clase.replace('_', ' ')
    return f"""Eres un radiólogo experto en neuroimagen. Escribe un informe radiológico sintético de TC de cráneo.

PATOLOGÍA: {nombre}
DESCRIPCIÓN CLÍNICA: {desc}

INFORME DE REFERENCIA (imita su estilo y estructura, NO lo copies):
--- HALLAZGOS ---
{hallazgos_ejemplo}
--- OPINIÓN ---
{opinion_ejemplo}

INSTRUCCIONES:
1. Escribe un informe NUEVO y DIFERENTE con el mismo estilo clínico
2. DEBE documentar {nombre} de forma explícita con terminología radiológica
3. Varía: localización, magnitud, hallazgos asociados
4. Clínicamente plausible — no inventes anatomía imposible
5. NO copies frases del ejemplo

Responde ÚNICAMENTE con este JSON, sin texto adicional:
{{"hallazgos": "<texto>", "opinion": "<texto>"}}"""


class RateLimiter:
    """Rate limiter thread-safe. Garantiza máximo `rpm` requests/minuto."""
    def __init__(self, rpm: int):
        self._interval = 60.0 / max(rpm, 1)
        self._lock     = threading.Lock()
        self._last     = 0.0
        print(f"RateLimiter: {rpm} RPM → {self._interval:.2f}s entre requests")

    def wait(self, stop_event=None):
        with self._lock:
            gap = self._interval - (time.monotonic() - self._last)
            if gap > 0:
                if stop_event:
                    stop_event.wait(timeout=gap)
                else:
                    time.sleep(gap)
            self._last = time.monotonic()


def generar_reporte(hallazgos_ej: str, opinion_ej: str, clase: str,
                    retries: int = 2) -> dict | None:
    """
    Llama a Claude API. Retorna {'hallazgos': ..., 'opinion': ...} o None.
    El rate limiter se aplica FUERA de esta función (en el worker).
    """
    prompt = construir_prompt(hallazgos_ej, opinion_ej, clase)

    for attempt in range(retries + 1):
        try:
            msg = claude.messages.create(
                model      = MODELO,
                max_tokens = 600,
                temperature= TEMPERATURA,
                messages   = [{"role": "user", "content": prompt}]
            )
            texto = msg.content[0].text.strip()

            # Extraer JSON aunque haya texto envolvente
            ini = texto.find('{')
            fin = texto.rfind('}')
            if ini == -1 or fin <= ini:
                return None

            datos = json.loads(texto[ini:fin+1])
            h = str(datos.get('hallazgos', datos.get('Hallazgos', ''))).strip()
            o = str(datos.get('opinion',   datos.get('Opinión', datos.get('Opinion', '')))).strip()

            if not h or not o:
                return None
            return {'hallazgos': h, 'opinion': o}

        except json.JSONDecodeError:
            return None
        except anthropic.RateLimitError:
            wait = 60 * (attempt + 1)
            print(f"\n  ⚠ RateLimit 429 — esperando {wait}s...", flush=True)
            time.sleep(wait)
        except anthropic.APIError as e:
            if attempt < retries:
                time.sleep(2 ** attempt)
            else:
                print(f"\n  ✗ APIError: {e}", flush=True)
                return None
    return None


rate_limiter = RateLimiter(rpm=RPM_LIMIT)

# ── Prueba de conexión ────────────────────────────────────────────
print("\nPrueba de conexión con Claude API...")
clase_prueba   = 'fractura_compleja_craneo'
ejemplo_prueba = df[df[clase_prueba] == 1].iloc[0]
res_prueba     = generar_reporte(ejemplo_prueba['Hallazgos'],
                                  ejemplo_prueba['Opinión'],
                                  clase_prueba)
if res_prueba:
    print("✓ Claude responde correctamente")
    print(f"  Hallazgos ({len(res_prueba['hallazgos'].split())} palabras): {res_prueba['hallazgos'][:140]}...")
    print(f"  Opinión   ({len(res_prueba['opinion'].split())} palabras): {res_prueba['opinion'][:100]}...")
else:
    print("✗ Sin respuesta — verifica ANTHROPIC_API_KEY y MODELO")

RateLimiter: 25 RPM → 2.40s entre requests

Prueba de conexión con Claude API...

  ✗ APIError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CaXGKbnAD9JMCnF5KfuKn'}
✗ Sin respuesta — verifica ANTHROPIC_API_KEY y MODELO


## 5 · Validador de reportes

In [8]:
def validar_reporte(reporte: dict, clase: str,
                    vectorizer_ref=None, matrix_ref=None) -> tuple[bool, str]:
    """
    6 criterios en orden de coste computacional (primero los más baratos).
    Retorna (valido, motivo_rechazo).
    """
    h = reporte.get('hallazgos', '')
    o = reporte.get('opinion', '')
    t = (h + ' ' + o).lower()

    # 1. Keyword de la patología presente
    if not any(kw in t for kw in KEYWORDS[clase]):
        return False, 'sin_keyword'

    # 2. Hallazgos mínimos (< 15 palabras = respuesta truncada)
    if len(h.split()) < 15:
        return False, 'hallazgos_cortos'

    # 3. JSON no completado (placeholder visible)
    if h.strip().startswith('<texto') or o.strip().startswith('<texto'):
        return False, 'formato_incorrecto'

    # 4. Frases meta del asistente
    if any(f in t for f in META_FRASES):
        return False, 'meta_frase'

    # 5. Opinión más larga que hallazgos (campos invertidos)
    if len(h) <= len(o):
        return False, 'campos_invertidos'

    # 6. Similitud TF-IDF con reales (costoso — va al final)
    if vectorizer_ref is not None and matrix_ref is not None:
        v = vectorizer_ref.transform([h + ' ' + o])
        if cosine_similarity(v, matrix_ref)[0].max() >= UMBRAL_SIMILITUD:
            return False, 'similar_a_real'

    return True, 'OK'


# Probar validador con el reporte de prueba
if res_prueba:
    ok, motivo = validar_reporte(res_prueba, clase_prueba)
    print(f"Validador sobre ejemplo de prueba: {'✓ válido' if ok else f'✗ {motivo}'}")
print(f"Umbral TF-IDF configurado: {UMBRAL_SIMILITUD}")

Umbral TF-IDF configurado: 0.9


## 6 · Motor de generación con checkpoints y monitoreo en tiempo real

In [9]:
def _guardar_checkpoint(sinteticos: list, ruta: Path) -> bool:
    """Guarda y verifica el checkpoint. Retorna True si OK."""
    try:
        with open(ruta, 'w', encoding='utf-8') as f:
            json.dump(sinteticos, f, ensure_ascii=False, indent=2)
        # Verificación de escritura
        with open(ruta, encoding='utf-8') as f:
            leidos = json.load(f)
        if len(leidos) != len(sinteticos):
            print(f"\n  ⚠ Checkpoint corrupto: escribió {len(sinteticos)}, leyó {len(leidos)}")
            return False
        return True
    except Exception as e:
        print(f"\n  ⚠ Error guardando checkpoint {ruta}: {e}")
        return False


def augmentar_clase(
    df_original:      pd.DataFrame,
    clase:            str,
    n_sinteticos:     int,
    ruta_checkpoint:  Path,
    vectorizer_ref,
    matrix_ref,
    rate_limiter:     RateLimiter,
    n_workers:        int = 2,
) -> tuple[pd.DataFrame, dict]:
    """
    Genera n_sinteticos reportes para 'clase'.
    — Checkpoint cada 25 reportes aceptados
    — Log en tiempo real línea por línea
    — Retoma desde checkpoint si existe
    — Retorna (df_sinteticos, dict_rechazos)
    """
    ejemplos = df_original[df_original[clase] == 1].reset_index(drop=True)
    tag      = clase[:26].upper()

    print(f"\n{'━'*65}")
    print(f"  {tag}")
    print(f"  Reales: {len(ejemplos)}  |  A generar: {n_sinteticos}  |  Workers: {n_workers}")
    print(f"  Checkpoint: {ruta_checkpoint.resolve()}")
    print(f"{'━'*65}")

    # ── Retomar desde checkpoint ──────────────────────────────────
    sinteticos: list = []
    if ruta_checkpoint.exists():
        try:
            with open(ruta_checkpoint, encoding='utf-8') as f:
                sinteticos = json.load(f)
            print(f"  ↺ Checkpoint encontrado: {len(sinteticos)} reportes previos cargados")
        except Exception as e:
            print(f"  ⚠ Checkpoint corrupto ({e}) — empezando de cero")
            sinteticos = []

    ya_generados = len(sinteticos)
    pendientes   = n_sinteticos - ya_generados

    if pendientes <= 0:
        print(f"  ✓ Ya completado ({ya_generados}/{n_sinteticos}) — nada que generar")
        df_s = pd.DataFrame(sinteticos)
        for p in PATOLOGIAS:
            if p not in df_s.columns: df_s[p] = 0
        return df_s, {}

    print(f"  Pendientes: {pendientes}  (ya generados: {ya_generados})")
    print()

    # ── Estado compartido ─────────────────────────────────────────
    rechazos  = Counter()
    lock      = threading.Lock()
    nuevos    = [0]
    intentos  = [0]
    stop      = threading.Event()
    t_inicio  = time.monotonic()

    CHECKPOINT_CADA = 25  # guardar cada N aceptados

    def _log_linea(final=False):
        total    = ya_generados + nuevos[0]
        pct      = 100 * total / n_sinteticos
        elapsed  = time.monotonic() - t_inicio
        vel      = nuevos[0] / elapsed * 60 if elapsed > 0 else 0
        eta_min  = (pendientes - nuevos[0]) / vel if vel > 0 else float('inf')
        rechazos_str = ' | '.join(f"{k}:{v}" for k,v in sorted(rechazos.items())) or '-'
        ts = datetime.now().strftime('%H:%M:%S')
        linea = (f"  [{ts}] {total:3d}/{n_sinteticos} ({pct:4.0f}%)  "
                 f"intentos:{intentos[0]:4d}  "
                 f"vel:{vel:4.1f}/min  "
                 f"eta:{eta_min:4.0f}min  "
                 f"rechazos:[{rechazos_str}]")
        if final:
            print(linea)
        else:
            print(linea)

    def _worker():
        while not stop.is_set():
            with lock:
                if nuevos[0] >= pendientes:
                    return

            rate_limiter.wait(stop_event=stop)
            if stop.is_set():
                return

            ejemplo = ejemplos.sample(1).iloc[0]
            reporte = generar_reporte(ejemplo['Hallazgos'], ejemplo['Opinión'], clase)

            with lock:
                intentos[0] += 1

            if reporte is None:
                with lock:
                    rechazos['api_error'] += 1
                continue

            valido, motivo = validar_reporte(reporte, clase, vectorizer_ref, matrix_ref)

            with lock:
                if not valido:
                    rechazos[motivo] += 1
                    continue

                if nuevos[0] >= pendientes:
                    return

                nuevos[0] += 1
                sinteticos.append({
                    'Hallazgos': reporte['hallazgos'],
                    'Opinión':   reporte['opinion'],
                    'tipo':      'sintetico',
                    clase:       1
                })

                # Log cada nuevo aceptado
                _log_linea(final=(nuevos[0] >= pendientes))

                # Checkpoint cada CHECKPOINT_CADA aceptados
                hacer_ckpt   = (nuevos[0] % CHECKPOINT_CADA == 0) or (nuevos[0] >= pendientes)
                snap         = list(sinteticos) if hacer_ckpt else None
                terminar     = nuevos[0] >= pendientes

            # I/O fuera del lock
            if snap is not None:
                ok = _guardar_checkpoint(snap, ruta_checkpoint)
                if ok:
                    print(f"  💾 Checkpoint guardado: {len(snap)} reportes → {ruta_checkpoint.name}")
                else:
                    print(f"  ⚠ Falló el checkpoint en {len(snap)} reportes")
            if terminar:
                stop.set()
                return

    # ── Lanzar workers ────────────────────────────────────────────
    with ThreadPoolExecutor(max_workers=n_workers,
                            thread_name_prefix=f"{clase[:10]}_w") as pool:
        futuros = [pool.submit(_worker) for _ in range(n_workers)]
        for f in as_completed(futuros):
            try:
                f.result()
            except Exception as e:
                print(f"  ✗ Worker error: {e}")

    # ── Resumen final de clase ────────────────────────────────────
    elapsed = time.monotonic() - t_inicio
    total_final = ya_generados + nuevos[0]
    print(f"\n  {'─'*60}")
    print(f"  ✓ {tag} completado")
    print(f"    Aceptados esta sesión : {nuevos[0]}")
    print(f"    Total en checkpoint   : {total_final}")
    print(f"    Intentos API          : {intentos[0]}")
    print(f"    Tasa de aceptación    : {100*nuevos[0]/max(intentos[0],1):.1f}%")
    print(f"    Tiempo                : {elapsed/60:.1f} min")
    if rechazos:
        print(f"    Rechazos por motivo:")
        for motivo, cnt in sorted(rechazos.items(), key=lambda x: -x[1]):
            print(f"      {motivo:<30} {cnt}")

    df_sint = pd.DataFrame(sinteticos)
    for p in PATOLOGIAS:
        if p not in df_sint.columns:
            df_sint[p] = 0

    return df_sint, dict(rechazos)


print("Motor de generación listo.")

Motor de generación listo.


## 7 · Ejecución — ambas clases en paralelo

Si se interrumpe, vuelve a ejecutar esta celda — retoma desde el checkpoint de cada clase.

In [ ]:
checkpoint_dir = Path('../data/augmentation_checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)
print(f"Checkpoint dir: {checkpoint_dir.resolve()}")

# ── Índices TF-IDF (solo reales, para el validador de similitud) ──
print("\nConstruyendo índices TF-IDF de referencia...")
vectorizers_ref, matrices_ref = {}, {}
for clase in CLASES_A_AUGMENTAR:
    textos = (df[df[clase]==1]['Hallazgos'] + ' ' + df[df[clase]==1]['Opinión']).tolist()
    vec    = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
    mat    = vec.fit_transform(textos)
    vectorizers_ref[clase] = vec
    matrices_ref[clase]    = mat
    print(f"  {clase}: {mat.shape[0]} ejemplos reales indexados")

t0 = time.monotonic()
dataframes_sinteticos: dict = {}
stats_rechazos:        dict = {}

def _procesar_clase(clase):
    ckpt = checkpoint_dir / f"checkpoint_{clase}.json"
    df_s, rech = augmentar_clase(
        df_original     = df,
        clase           = clase,
        n_sinteticos    = N_SINTETICOS_POR_CLASE,
        ruta_checkpoint = ckpt,
        vectorizer_ref  = vectorizers_ref[clase],
        matrix_ref      = matrices_ref[clase],
        rate_limiter    = rate_limiter,
        n_workers       = 2,
    )
    return clase, df_s, rech

print(f"\nLanzando {len(CLASES_A_AUGMENTAR)} clases en paralelo...")
print("="*65)

with ThreadPoolExecutor(max_workers=len(CLASES_A_AUGMENTAR),
                        thread_name_prefix="clase_outer") as pool:
    futuros = {pool.submit(_procesar_clase, c): c for c in CLASES_A_AUGMENTAR}
    for fut in as_completed(futuros):
        clase_key = futuros[fut]
        try:
            nombre, df_s, rech = fut.result()
            dataframes_sinteticos[nombre] = df_s
            stats_rechazos[nombre]        = rech
        except Exception as e:
            print(f"\n  ✗ {clase_key}: Error fatal — {e}")
            dataframes_sinteticos[clase_key] = pd.DataFrame()
            stats_rechazos[clase_key]        = {}

elapsed = time.monotonic() - t0
print(f"\n{'═'*65}")
print(f"  Generación completada en {elapsed/60:.1f} min")
for clase in CLASES_A_AUGMENTAR:
    n = len(dataframes_sinteticos.get(clase, []))
    print(f"  {clase:<32} → {n} sintéticos")

Checkpoint dir: C:\Users\Isabella\Documents\ICESI\8. Octavo Semestre\Proyecto de Grado I\PDG-Diagnostico-Clinico\data\augmentation_checkpoints

Construyendo índices TF-IDF de referencia...
  desviacion_linea_media: 129 ejemplos reales indexados
  fractura_compleja_craneo: 30 ejemplos reales indexados

Lanzando 2 clases en paralelo...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FRACTURA_COMPLEJA_CRANEO
  Reales: 30  |  A generar: 300  |  Workers: 2

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  DESVIACION_LINEA_MEDIA
  Reales: 129  |  A generar: 300  |  Workers: 2
  Checkpoint: C:\Users\Isabella\Documents\ICESI\8. Octavo Semestre\Proyecto de Grado I\PDG-Diagnostico-Clinico\data\augmentation_checkpoints\checkpoint_fractura_compleja_craneo.json
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Checkpoint: C:\Users\Isabella\Documents\ICESI\8. Octavo Semestre\Proyecto de Grado I\PDG-Diagnostico-Clinico\data\augmentation_checkp

## 8 · Dataset aumentado y visualización

In [9]:
# Combinar reales + sintéticos
df_real = df.copy()
df_real['tipo'] = 'real'

partes = [df_real]
for clase, df_s in dataframes_sinteticos.items():
    if len(df_s) > 0:
        partes.append(df_s)

df_aumentado = pd.concat(partes, ignore_index=True)
for p in PATOLOGIAS:
    if p not in df_aumentado.columns: df_aumentado[p] = 0
df_aumentado[PATOLOGIAS] = df_aumentado[PATOLOGIAS].fillna(0).astype(int)

print("DATASET AUMENTADO")
print("=" * 65)
print(f"Total: {len(df_aumentado)}  ({(df_aumentado['tipo']=='real').sum()} reales + "
      f"{(df_aumentado['tipo']=='sintetico').sum()} sintéticos)")
print()
print(f"  {'Clase':<30} {'Reales':>8} {'Sint.':>8} {'Total':>8} {'%':>6}")
print("  " + "-"*60)
for p in PATOLOGIAS:
    nr = df_aumentado[(df_aumentado['tipo']=='real') & (df_aumentado[p]==1)].shape[0]
    ns = df_aumentado[(df_aumentado['tipo']=='sintetico') & (df_aumentado[p]==1)].shape[0]
    tot = nr + ns
    print(f"  {p:<30} {nr:>8} {ns:>8} {tot:>8} {100*tot/len(df_aumentado):>5.1f}%")

NameError: name 'dataframes_sinteticos' is not defined

In [ ]:
# Visualización antes / después
colores = ['#e74c3c','#3498db','#f39c12','#2ecc71']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

antes = [int(df[p].sum()) for p in PATOLOGIAS]
axes[0].bar(range(len(PATOLOGIAS)), antes, color=colores, alpha=0.8, edgecolor='black')
axes[0].set_title('ANTES del augmentation', fontweight='bold')
axes[0].set_ylabel('Casos positivos')
axes[0].set_yscale('log')
axes[0].set_xticks(range(len(PATOLOGIAS)))
axes[0].set_xticklabels([p.replace('_','\n') for p in PATOLOGIAS], fontsize=8)
for i, v in enumerate(antes):
    axes[0].text(i, v*1.2, str(v), ha='center', fontsize=9, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

x = range(len(PATOLOGIAS))
dr = [df_aumentado[(df_aumentado['tipo']=='real')&(df_aumentado[p]==1)].shape[0] for p in PATOLOGIAS]
ds = [df_aumentado[(df_aumentado['tipo']=='sintetico')&(df_aumentado[p]==1)].shape[0] for p in PATOLOGIAS]
axes[1].bar(x, dr, color=colores, alpha=0.8, edgecolor='black', label='Reales')
axes[1].bar(x, ds, bottom=dr, color=colores, alpha=0.35,
            edgecolor='black', linestyle='--', linewidth=1, label='Sintéticos')
axes[1].set_title('DESPUÉS del augmentation', fontweight='bold')
axes[1].set_ylabel('Casos positivos')
axes[1].set_yscale('log')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels([p.replace('_','\n') for p in PATOLOGIAS], fontsize=8)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Efecto del augmentation LLM por clase', fontsize=13, fontweight='bold')
plt.tight_layout()
plot_path = Path('../data/augmentation_comparacion.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Gráfico guardado: {plot_path.resolve()}")

## 9 · Inspección manual de calidad

Revisa pares real/sintético para documentar plausibilidad clínica en la tesis.

In [ ]:
def inspeccionar(clase: str, n: int = 3):
    print(f"\n{'═'*70}")
    print(f"  INSPECCIÓN: {clase.upper()}")
    print(f"{'═'*70}")
    reales = df[df[clase]==1].sample(min(n, int(df[clase].sum())), random_state=42)
    df_s   = dataframes_sinteticos.get(clase, pd.DataFrame())
    if len(df_s) == 0:
        print("  Sin sintéticos generados.")
        return
    sints = df_s.sample(min(n, len(df_s)), random_state=42)
    for i, (_, r) in enumerate(reales.iterrows()):
        s = sints.iloc[i]
        print(f"\n  ── Par {i+1} ──")
        print(f"  [REAL]      H: {r['Hallazgos'][:180]}")
        print(f"              O: {r['Opinión'][:120]}")
        print(f"  [SINTÉTICO] H: {s['Hallazgos'][:180]}")
        print(f"              O: {s['Opinión'][:120]}")

for clase in CLASES_A_AUGMENTAR:
    inspeccionar(clase, n=2)

## 10 · Diversidad léxica

In [ ]:
def analizar_diversidad(df_sint: pd.DataFrame, clase: str):
    if len(df_sint) < 2:
        print(f"  {clase}: insuficientes ejemplos")
        return
    textos = (df_sint['Hallazgos'] + ' ' + df_sint['Opinión']).tolist()
    vec    = TfidfVectorizer(max_features=500, ngram_range=(1,2))
    X      = vec.fit_transform(textos)
    n      = min(100, len(textos))
    idx    = np.random.choice(len(textos), n, replace=False)
    sim    = cosine_similarity(X[idx])
    np.fill_diagonal(sim, 0)
    prom   = sim.sum() / (n * (n-1))
    nivel  = ("✓ BUENA" if prom < 0.25 else
               "~ ACEPTABLE" if prom < 0.45 else
               "✗ BAJA — sube temperatura")
    print(f"  {clase:<30} sim_coseno_prom={prom:.3f}  {nivel}")

print("DIVERSIDAD LÉXICA DE LOS SINTÉTICOS")
print("=" * 65)
for clase, df_s in dataframes_sinteticos.items():
    if len(df_s) > 0:
        analizar_diversidad(df_s, clase)

## 11 · Split y exportación

**Test set = 100% datos reales** (principio del paper).

In [ ]:
df_reales  = df_aumentado[df_aumentado['tipo']=='real'].copy()
df_sints   = df_aumentado[df_aumentado['tipo']=='sintetico'].copy()

reales_train, df_test = train_test_split(
    df_reales,
    test_size = 0.15,
    stratify  = df_reales['fractura_compleja_craneo'],
    random_state = RANDOM_SEED
)
df_train = pd.concat([reales_train, df_sints], ignore_index=True)
df_train = df_train.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
df_test  = df_test.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print("SPLITS FINALES")
print("=" * 65)
print(f"Train: {len(df_train)}  "
      f"({(df_train['tipo']=='real').sum()} reales + "
      f"{(df_train['tipo']=='sintetico').sum()} sint.)")
print(f"Test:  {len(df_test)}  (100% reales)")
print()
print(f"  {'Clase':<30} {'Train pos':>10} {'Test pos':>10}")
print("  " + "-"*52)
for p in PATOLOGIAS:
    nt = int(df_train[p].sum())
    nv = int(df_test[p].sum())
    print(f"  {p:<30} {nt:>10} {nv:>10}")

In [ ]:
output_path = Path('../data/train_augmentado_llm.xlsx')
output_path.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_train.to_excel(writer, sheet_name='Train', index=False)
    df_test.to_excel(writer, sheet_name='Test', index=False)
    df_aumentado.to_excel(writer, sheet_name='Dataset_Completo', index=False)

print(f"Guardado: {output_path.resolve()}")
print(f"  Train            → {len(df_train)} filas")
print(f"  Test             → {len(df_test)} filas  (solo reales)")
print(f"  Dataset_Completo → {len(df_aumentado)} filas")

## 12 · Resumen final para la tesis

In [ ]:
print("=" * 65)
print("RESUMEN — DATA AUGMENTATION CON LLM")
print("=" * 65)
print(f"Modelo: {MODELO}  |  Temperatura: {TEMPERATURA}")
print(f"Método: imitación de estilo (Veselovsky et al., 2023 — Stanford HAI)")
print()

total_gen = total_rech = 0
rechazos_globales = Counter()

for clase in CLASES_A_AUGMENTAR:
    df_s   = dataframes_sinteticos.get(clase, pd.DataFrame())
    rech   = stats_rechazos.get(clase, {})
    n_real = int(df[clase].sum())
    n_sint = len(df_s)
    n_rech = sum(rech.values())
    total_gen  += n_sint
    total_rech += n_rech
    for k, v in rech.items(): rechazos_globales[k] += v

    tasa = 100 * n_sint / max(n_sint + n_rech, 1)
    print(f"Clase: {clase}")
    print(f"  Reales originales    : {n_real}")
    print(f"  Sintéticos aceptados : {n_sint}")
    print(f"  Rechazados           : {n_rech}  (tasa aceptación: {tasa:.1f}%)")
    for motivo, cnt in sorted(rech.items(), key=lambda x: -x[1]):
        print(f"    {motivo:<32} {cnt}")
    print()

print(f"TOTALES")
print(f"  Aceptados  : {total_gen}")
print(f"  Rechazados : {total_rech}")
print()
print(f"Split de evaluación:")
print(f"  Train: {len(df_train)} (reales + sint.)  |  Test: {len(df_test)} (solo reales)")
print()
print(f"Archivo: {output_path.resolve()}")